# Environment Setup

Configure a reproducible Python environment, load secrets from `.env`, and verify OpenAI API connectivity from Jupyter (Cursor / VS Code).


## 1. Overview

This notebook covers:

- Creating a Python 3.12+ virtual environment and installing dependencies from `requirements.txt`
- Storing `OPENAI_API_KEY` in a gitignored `.env` file (never hardcode secrets)
- Selecting the project interpreter as the Jupyter kernel
- Running local health checks (runtime, packages, `.env`) without calling the API
- Sending a minimal Chat Completions request to confirm end-to-end connectivity


## 2. Motivation

Downstream notebooks and scripts depend on a consistent interpreter, installed packages, and a valid API credential. Failures that look like "the API is broken" are often an inactive venv, the wrong Jupyter kernel, a missing `.env`, or a placeholder key.

The standard pattern:

1. Isolate dependencies in a **virtual environment**
2. Keep secrets in **`.env`**, loaded via `python-dotenv`
3. Run notebooks against that same interpreter as the **Jupyter kernel**


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **Virtual environment (venv)** | Isolated Python install and packages for this project |
| **`requirements.txt`** | Dependency list installed with `pip install -r requirements.txt` |
| **API key** | Credential for OpenAI; treat like a password |
| **`.env` file** | Local `KEY=value` file loaded into process environment variables |
| **`python-dotenv`** | Loads `.env` via `load_dotenv()` for `os.getenv(...)` |
| **Jupyter kernel** | Interpreter that executes notebook cells; must match project `.venv` |
| **OpenAI client** | `OpenAI()` reads `OPENAI_API_KEY` from the environment |

### 3.2 Runtime flow

1. Create `.venv` with `python -m venv .venv` and activate it.
2. Install packages from `requirements.txt`.
3. Copy `.env.example` → `.env` and set a real `OPENAI_API_KEY`.
4. Call `load_dotenv()` so variables from `.env` are available to the process.
5. Instantiate `OpenAI()`; it authenticates using `OPENAI_API_KEY`.
6. Ensure Cursor / VS Code uses the same `.venv` interpreter as the notebook kernel.

### 3.3 When to use this pattern

**Use for:** local development, demos, and services that need reproducible deps plus secrets.

**Avoid:** committing `.env`; pasting keys into chat or screenshots; sharing one long-lived key across teams without rotation; leaving secrets in notebook outputs.

**Production alternatives:** secret managers (AWS Secrets Manager, Azure Key Vault, GitHub Actions secrets) or host-injected environment variables — still never hardcode keys in source.


## 4. Architecture

### Setup flow

```mermaid
flowchart TD
    py[Python 3.12+] --> venv[Create and activate .venv]
    venv --> pip[pip install -r requirements.txt]
    pip --> env[Copy .env.example to .env]
    env --> key[Set OPENAI_API_KEY]
    key --> kernel[Select .venv as Jupyter kernel]
    kernel --> load[load_dotenv + OpenAI client]
    load --> ok[Health checks and first API call]
```

### Secret layout

```text
Project root
├── .env.example     ← template (safe to commit)
├── .env             ← real key (gitignored — never commit)
├── requirements.txt
└── notebooks/
    └── Environment_Setup.ipynb
            │
            ├─ load_dotenv()  → reads .env into os.environ
            └─ OpenAI()       → uses OPENAI_API_KEY
```


## 5. Installation

The validation cells below need **no API key**. They check the interpreter, package imports, and `.env` presence without printing secrets.

### 5.1 One-time terminal setup

From the **project root**:

```bash
python -m venv .venv

# Windows (PowerShell)
.\.venv\Scripts\Activate.ps1

# macOS / Linux
source .venv/bin/activate

pip install -r requirements.txt

# Optional: register a named Jupyter kernel
python -m ipykernel install --user --name=project-venv --display-name="Project (.venv)"

# Create .env from the template
# Windows:  Copy-Item .env.example .env
# macOS/Linux:  cp .env.example .env
```

If `pip install` fails on Windows with `No such file or directory` / long-path errors, enable Win32 long paths, or install packages from `requirements.txt` in smaller batches. Cursor / VS Code only needs `.venv` + `ipykernel` to execute notebooks.

Edit `.env` and set a real key from [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys):

```text
OPENAI_API_KEY=sk-...
```

In Cursor / VS Code: open this notebook → **Select Kernel** → choose `.venv` (or **Project (.venv)**).


In [1]:
# Local environment health check — no API call required
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )


REQUIRED_PACKAGES = [
    "openai",
    "dotenv",  # import name for python-dotenv
    "tiktoken",
    "pandas",
    "numpy",
]

root = find_project_root()
env_path = root / ".env"
env_example = root / ".env.example"

print("=== Environment health check ===")
print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version.split()[0]}")
print(f"Project root      : {root}")
print(f".env exists       : {env_path.is_file()}")
print(f".env.example      : {env_example.is_file()}")
print()

major, minor = sys.version_info[:2]
if (major, minor) < (3, 12):
    print(f"WARNING: Python 3.12+ recommended. Current: {major}.{minor}.")
else:
    print("OK: Python version meets 3.12+ requirement.")

print("\nPackage imports:")
missing: list[str] = []
for name in REQUIRED_PACKAGES:
    spec = importlib.util.find_spec(name)
    status = "OK" if spec is not None else "MISSING"
    print(f"  {name:<12} {status}")
    if spec is None:
        missing.append(name)

if missing:
    print("\nInstall dependencies from the project root:")
    print("  pip install -r requirements.txt")
else:
    print("\nOK: Required packages are importable in this kernel.")

if not env_path.is_file():
    print("\nNext: copy .env.example to .env and set OPENAI_API_KEY.")


=== Environment health check ===
Python executable : C:\Users\imrat\Downloads\Agentic_AI_Module\Setup_and_Prompt_Engineering\.venv\Scripts\python.exe
Python version    : 3.13.5
Project root      : C:\Users\imrat\Downloads\Agentic_AI_Module\Setup_and_Prompt_Engineering
.env exists       : True
.env.example      : True

OK: Python version meets 3.12+ requirement.

Package imports:
  openai       OK
  dotenv       OK
  tiktoken     OK
  pandas       OK
  numpy        OK

OK: Required packages are importable in this kernel.


In [2]:
# Inspect OPENAI_API_KEY presence without printing the secret
from pathlib import Path

from dotenv import load_dotenv
import os


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root (requirements.txt).")


root = find_project_root()
# Explicit path is more reliable when the notebook cwd is notebooks/
load_dotenv(root / ".env")

api_key = os.getenv("OPENAI_API_KEY", "")
placeholder = "your_openai_api_key"


def key_status(value: str) -> str:
    if not value.strip():
        return "MISSING — set OPENAI_API_KEY in .env"
    if placeholder in value.lower() or value.strip() == "your_openai_api_key_here":
        return "PLACEHOLDER — replace the example value with a real key"
    if not value.startswith("sk-"):
        return "UNEXPECTED FORMAT — OpenAI keys usually start with sk-"
    return f"SET — length {len(value)} chars (value hidden)"


print(f"Loaded .env from : {root / '.env'}")
print(f"OPENAI_API_KEY   : {key_status(api_key)}")


Loaded .env from : C:\Users\imrat\Downloads\Agentic_AI_Module\Setup_and_Prompt_Engineering\.env
OPENAI_API_KEY   : SET — length 164 chars (value hidden)


## 6. OpenAI Connectivity

With a real key in `.env` and the project kernel selected, the next cell sends a minimal Chat Completions request.

Standard client bootstrap:

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()          # or load_dotenv(project_root / ".env")
client = OpenAI()      # reads OPENAI_API_KEY from the environment
```


In [3]:
# Live OpenAI connectivity check — requires a valid OPENAI_API_KEY in .env
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI, AuthenticationError, APIConnectionError, RateLimitError
import os


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root (requirements.txt).")


root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()


def run_connectivity_check() -> None:
    """Confirm auth and network with a minimal chat completion."""
    api_key = os.getenv("OPENAI_API_KEY", "")
    if not api_key or "your_openai_api_key" in api_key.lower():
        print("OPENAI_API_KEY not configured.")
        print("Copy .env.example to .env at the project root and set your key.")
        return

    messages = [
        {
            "role": "system",
            "content": "You are a concise assistant. Reply in one short sentence.",
        },
        {
            "role": "user",
            "content": (
                "Confirm that the OpenAI environment is working by saying: "
                "Environment setup OK."
            ),
        },
    ]

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=40,
        )
        text = response.choices[0].message.content
        print("API call succeeded.")
        print(f"Model : {response.model}")
        print(f"Reply : {text}")
        if response.usage:
            print(
                f"Tokens: prompt={response.usage.prompt_tokens}, "
                f"completion={response.usage.completion_tokens}"
            )
    except AuthenticationError:
        print("Authentication failed. Check that OPENAI_API_KEY is valid and not revoked.")
    except RateLimitError:
        print("Rate limit or quota hit. Wait and retry, or check billing/limits.")
    except APIConnectionError as exc:
        print(f"Network error reaching OpenAI: {exc}")
    except Exception as exc:
        print(f"Request failed: {type(exc).__name__}: {exc}")


run_connectivity_check()


API call succeeded.
Model : gpt-4o-mini-2024-07-18
Reply : Environment setup OK.
Tokens: prompt=38, completion=4


## 7. Implementation notes

1. **`find_project_root`** — Notebooks often run with cwd = `notebooks/`. Walking up for `requirements.txt` resolves the project root so `.env` and dependencies load correctly.
2. **Health check** — Uses `sys.executable` / `sys.version` and `importlib.util.find_spec` to confirm the kernel matches the project venv, without spending API credits.
3. **Key status** — Calls `load_dotenv(root / ".env")`, then reports MISSING / PLACEHOLDER / SET **without printing the key**.
4. **Connectivity check** — Issues a minimal `chat.completions.create(...)` call and surfaces auth, rate-limit, and network errors separately.
5. **Load order** — Call `load_dotenv` before `OpenAI()` so the key is present when the client initializes.


## 8. Best practices

- Keep one venv per project; activate it before installing or running notebooks.
- Store secrets only in `.env` (or a secret manager); never in source, notebooks, or screenshots.
- Commit `.env.example`, not `.env`. Keep `.env` in `.gitignore`.
- Select the project `.venv` as the Jupyter kernel before running cells.
- Prefer an explicit `.env` path from the project root when notebooks live in a subfolder.
- Validate locally first; only then spend tokens on a live API call.
- Rotate and revoke keys immediately if they leak.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `ModuleNotFoundError` in notebook after `pip install` | Kernel ≠ venv where packages were installed | Select `.venv` as the kernel / interpreter |
| `AuthenticationError` | Missing, placeholder, or revoked key | Set a real `OPENAI_API_KEY` in project-root `.env` |
| Key "not found" though `.env` exists | `.env` under `notebooks/` or cwd mismatch | Place `.env` next to `.env.example`; load with absolute path |
| Packages install but imports still fail | `pip` ran against system Python | Activate `.venv` first, confirm `sys.executable` |
| Stale imports after recreating `.venv` | Old kernel still selected | Reselect interpreter; reinstall `ipykernel` if needed |

Also avoid printing `os.getenv("OPENAI_API_KEY")` in shared notebooks or committing outputs that contain secrets.


## 10. Validation checklist

1. Run the health-check cell and confirm Python ≥ 3.12, packages OK, and `.env` present.
2. Run the key-status cell and confirm status is `SET` (not MISSING / PLACEHOLDER).
3. Run the connectivity cell and confirm a successful model reply.
4. Optionally harden checks with `assert_environment_ready` below.


In [4]:
# Aggregated readiness check
from __future__ import annotations

import importlib.util
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root (requirements.txt).")


def assert_environment_ready(root: Path) -> list[str]:
    """Return human-readable problems; empty list means ready."""
    problems: list[str] = []

    major, minor = sys.version_info[:2]
    if (major, minor) < (3, 12):
        problems.append(f"Python 3.12+ required; found {major}.{minor}")

    for pkg in ("openai", "dotenv"):
        if importlib.util.find_spec(pkg) is None:
            problems.append(
                f"Package not importable: {pkg} (pip install -r requirements.txt)"
            )

    env_file = root / ".env"
    if not env_file.is_file():
        problems.append(".env missing — copy .env.example to .env")
    else:
        load_dotenv(env_file)
        key = os.getenv("OPENAI_API_KEY", "")
        if not key.strip():
            problems.append("OPENAI_API_KEY is empty")
        elif "your_openai_api_key" in key.lower():
            problems.append("OPENAI_API_KEY is still the placeholder value")

    return problems


root = find_project_root()
issues = assert_environment_ready(root)
if not issues:
    print("Environment ready.")
else:
    print("Fix these issues:")
    for item in issues:
        print(f"  - {item}")


Environment ready.


## 11. Team onboarding (optional)

For a new engineer joining this repo, document:

- Exact commands for venv create/activate, `pip install -r requirements.txt`, and `.env` creation
- How to select the Jupyter kernel in Cursor / VS Code
- A verify step: run this notebook's health check + connectivity check
- A short troubleshooting table (wrong kernel, missing `.env`, placeholder key, auth error)

Do not include real API keys in that document.


## 12. Summary

- Use a **Python 3.12+ venv**, install from `requirements.txt`, and point Jupyter at that interpreter.
- Keep `OPENAI_API_KEY` in a gitignored **`.env`** file; load it with `python-dotenv` before creating `OpenAI()`.
- Validate locally (version, packages, key status), then run a minimal Chat Completions call.
- Most connectivity failures are kernel, path, or placeholder-key issues — not model problems.

**Next:** `OpenAI_SDK.ipynb` for OpenAI Python SDK usage patterns.
